In [13]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['band_2'] - df['band_1']) / (df['band_2'] + df['band_1'])
    return df["ndvi"]

In [8]:
def get_bands(df):
    return df[["SR_B4", "SR_B5"]]

In [ ]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()

In [6]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
image = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-06-01', '2018-09-30').map(prep_sr_l8).select(['SR_B4', 'SR_B5']).median().set({"system:time_start": ee.Date("2018-06-01").millis()})
ic = ee.ImageCollection([image])

In [9]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

chunks = {"time": 1, "X": 2048, "Y": 2048}

list_of_column_names = ["SR_B4", "SR_B5"]

run(
    dataset=ic,
    source="ee",
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": get_bands,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        "chunks": chunks,
        "output_column_names": list_of_column_names
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_Tiles",
        "vrt": True
    }
)

Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:49374' processes=112 threads=112, memory=713.97 GiB>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] AOI tiling enabled. Streaming tiles in batches...
[robustraster] Processing tile 1 of 15
[robustraster] Running user function...
Exported: Plumas_Tiles\x-116467_-89917_y151096_179986_tile_1__time_20180601T000000.tif with bands [np.str_('SR_B4'), np.str_('SR_B5')]
[robustraster] Processing tile 2 of 15
[robustraster] Running user function...
Exported: Plumas_Tiles\x-90087_-80727_y170804_179984_tile_2__time_20180601T000000.tif with bands [np.str_('SR_B4'), np.str_('SR_B5')]
[robustraster] Processing tile 3 of 15
[robustraster] Running user function...
Exported: Plumas_Tiles\x-145305_-119625_y182617_209977_tile_3__time_20180601T000000.tif with bands [np.str_('SR_B4'), np.str_('SR_B5')]
[robustraster] Processing tile 4 of 15
[robustraster] Run

r:\Users\adrianom.UNR\.conda\envs\rreditmode\lib\site-packages\osgeo\gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [14]:
from robustraster import run
import pandas as pd
import glob

chunks = {"time": 1, "X": 2048, "Y": 2048}

local_2018_files = glob.glob(r"R:\SCRATCH\adrianom\code\big-raster-helper-experiment-code\Plumas_Tiles\*.tif")
list_of_column_names = ["ndvi"]

run(
    dataset=r"./Plumas_Tiles/time_20180601T000000.vrt",
    source="local",
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        "chunks": chunks,
        "output_column_names": list_of_column_names
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Plumas_NDVI_Tiles",
    },
    dask_mode="custom",
    dask_config={
        "n_workers": 20,
        "threads_per_worker": 1,
        "memory_limit": "3g",
    },
)

Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:63216' processes=20 threads=20, memory=55.88 GiB>
[robustraster] Running user function...
Exported: Plumas_NDVI_Tiles\x-22455_-3675_y151125_194775__time_0.tif with bands [np.str_('ndvi')]
Exported: Plumas_NDVI_Tiles\x-145335_-83925_y194805_256215__time_0.tif with bands [np.str_('ndvi')]
Exported: Plumas_NDVI_Tiles\x-22455_-3675_y194805_256215__time_0.tif with bands [np.str_('ndvi')]
Exported: Plumas_NDVI_Tiles\x-145335_-83925_y151125_194775__time_0.tif with bands [np.str_('ndvi')]
Exported: Plumas_NDVI_Tiles\x-83895_-22485_y151125_194775__time_0.tif with bands [np.str_('ndvi')]
Exported: Plumas_NDVI_Tiles\x-83895_-22485_y194805_256215__time_0.tif with bands [np.str_('ndvi')]
[robustraster] Dask client closed.
[robustraster] Dask cluster shut down.
